# ATM Cash Demand - Time Series Model Training & Evaluation

## Overview
This notebook continues from `01_data_preprocess.ipynb` and implements:
1. **Feature Engineering**: Lags, Rolling statistics, Time features, UK Holidays
2. **Stationarity Testing**: ADF test for time series decomposition
3. **Model Training**:
   - ARIMA Baseline
   - SARIMAX with Exogenous features
   - Global LightGBM (1 model for all 111 ATMs)
4. **Model Evaluation**: SMAPE, MAE, RMSE, Bias metrics
5. **Visualization**: SHAP feature importance, Actual vs Predicted plots

## Input
- **data/processed/**: train_df.pkl, val_df.pkl, test_df.pkl (from 01_data_preprocess)

## Output
- **data/processed/**: Feature-engineered datasets, model predictions
- **report/figures/**: Model evaluation plots and SHAP explanations

## 1. Import Libraries & Load Processed Data

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import lightgbm as lgb
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import warnings
import os

warnings.filterwarnings('ignore')

# Set global styles
MAIN_BLUE = '#1f497d'
LIGHT_GREY = '#d9d9d9'
TEXT_GREY = '#595959'
AXIS_GREY = '#8c8c8c'
BG_HIGHLIGHT = '#f2f2f2'

sns.set_style('white')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = TEXT_GREY
plt.rcParams['axes.labelcolor'] = TEXT_GREY
plt.rcParams['xtick.color'] = TEXT_GREY
plt.rcParams['ytick.color'] = TEXT_GREY

# Ensure output directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../report/figures', exist_ok=True)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. Load Processed Data from 01_data_preprocess

In [14]:
# Load processed datasets
print("📂 Loading processed data from data/processed/...\n")

train_feat = pd.read_pickle('../data/processed/train_feat.pkl')
val_feat = pd.read_pickle('../data/processed/val_feat.pkl')
test_feat = pd.read_pickle('../data/processed/test_feat.pkl')
submission_df = pd.read_pickle('../data/processed/submission_df.pkl')

# Convert Date to datetime if needed
for df in [train_feat, val_feat, test_feat, submission_df]:
    df['Date'] = pd.to_datetime(df['Date'])

print(f"✅ Data loaded successfully")
print(f"\n📊 Data Summary:")
print(f"{'='*60}")
print(f"Train:      {train_feat.shape[0]:>8} rows | {train_feat['ATM_ID'].nunique()} ATMs")
print(f"Val:        {val_feat.shape[0]:>8} rows | {val_feat['ATM_ID'].nunique()} ATMs")
print(f"Test:       {test_feat.shape[0]:>8} rows | {test_feat['ATM_ID'].nunique()} ATMs")
print(f"Submission: {submission_df.shape[0]:>8} rows | {submission_df['ATM_ID'].nunique()} ATMs")
print(f"{'='*60}")

FORECAST_HORIZON = 56  # 56-day forecast horizon

📂 Loading processed data from data/processed/...

✅ Data loaded successfully

📊 Data Summary:
Train:         69153 rows | 111 ATMs
Val:            6216 rows | 111 ATMs
Test:           6216 rows | 111 ATMs
Submission:     6216 rows | 111 ATMs


## 4. Prepare Data Streams (Time Series & Machine Learning)

In [15]:
# =================================================================
# STREAM 1: For Statistical Models (ARIMA/SARIMAX)
# =================================================================
print("📈 Preparing data for Statistical Models (ARIMA/SARIMAX)...\n")
train_ts = train_feat.copy()

# Use linear interpolation for missing values
train_ts['Withdrawal'] = train_ts.groupby('ATM_ID')['Withdrawal'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

# =================================================================
# STREAM 2: For Machine Learning (LightGBM)
# =================================================================
print("🤖 Preparing data for Machine Learning (LightGBM)...\n")
train_ml = train_feat.copy()
train_ml = train_ml.dropna(subset=['Withdrawal'])

# =================================================================

print(f"✅ Data streams prepared")
print(f"\n📊 NaN Analysis:")
print(f"   Train (Time Series):  {train_ts['Withdrawal'].isna().sum()} NaN (interpolated)")
print(f"   Train (Machine Learn): {train_ml['Withdrawal'].isna().sum()} NaN (removed)")
print(f"   Val:                  {val_feat['Withdrawal'].isna().sum()} NaN")
print(f"   Test:                 {test_feat['Withdrawal'].isna().sum()} NaN")

📈 Preparing data for Statistical Models (ARIMA/SARIMAX)...

🤖 Preparing data for Machine Learning (LightGBM)...

✅ Data streams prepared

📊 NaN Analysis:
   Train (Time Series):  0 NaN (interpolated)
   Train (Machine Learn): 0 NaN (removed)
   Val:                  75 NaN
   Test:                 81 NaN


## 5. Stationarity Testing (Train-Only, No Data Leakage)

In [16]:
def check_stationarity_train_only(train_wide, significance=0.05):
    """
    Test stationarity ONLY on Train set to avoid data leakage.
    """
    stationary_info = {}
    
    print("🔍 Testing stationarity for 111 ATMs (Train-only)...\n")
    
    for atm in train_wide.columns:
        train_series = train_wide[atm].dropna()
        
        results = {'atm': atm}
        
        if len(train_series) < 20:
            results['train'] = False
            results['d'] = 1
        else:
            stat, pvalue, _, _, _, _ = adfuller(train_series, autolag='AIC')
            is_stationary = pvalue < significance
            
            results['train'] = is_stationary
            results['d'] = 0 if is_stationary else 1
            
        stationary_info[atm] = results
    
    # Statistics
    stationary_count = sum(1 for v in stationary_info.values() if v['train'])
    needs_diff_count = sum(1 for v in stationary_info.values() if v['d'] == 1)
    
    print(f"📊 Stationarity Test Results (Train-only):")
    print(f"   ✅ Stationary (d=0)     : {stationary_count} ATMs")
    print(f"   🔄 Need differencing (d=1) : {needs_diff_count} ATMs")
    
    return stationary_info

# Run stationarity test
train_wide = train_ts.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
stationarity_info = check_stationarity_train_only(train_wide)

print(f"\n✅ Stationarity testing complete\n")

🔍 Testing stationarity for 111 ATMs (Train-only)...

📊 Stationarity Test Results (Train-only):
   ✅ Stationary (d=0)     : 101 ATMs
   🔄 Need differencing (d=1) : 10 ATMs

✅ Stationarity testing complete



## 6. Model Evaluation Function

In [17]:
def smape_safe(y_true, y_pred):
    """
    Calculate SMAPE safely, handling NaN and division by zero.
    """
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    if np.sum(mask) == 0:
        return 0.0
    y_true, y_pred = y_true[mask], y_pred[mask]
    denom = np.abs(y_true) + np.abs(y_pred)
    return np.mean(np.where(denom == 0, 0, np.abs(y_true - y_pred) / denom)) * 200

def evaluate_model_comprehensive(model_name, train_df, val_df, test_df, predictions_dict=None):
    """
    Comprehensive model evaluation with SMAPE, MAE, RMSE, Bias metrics.
    """
    last_train_vals = train_df.dropna(subset=['Withdrawal']).groupby('ATM_ID')['Withdrawal'].last()
    
    val_wide = val_df.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
    test_wide = test_df.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
    
    print(f"\n=== {model_name} EVALUATION ===")
    print(f"{'Tập':<12} {'SMAPE':<8} {'MAE':<8} {'RMSE':<8} {'Bias':<8}")
    print("-" * 55)
    
    for name, df_wide in [('Val', val_wide), ('Test', test_wide)]:
        smape_list, mae_list, rmse_list, bias_list = [], [], [], []
        
        horizon = len(df_wide)
        
        for atm in df_wide.columns:
            actual = df_wide[atm].values
            
            if predictions_dict is not None and atm in predictions_dict:
                pred = np.array(predictions_dict[atm])
                pred = pred[-horizon:] if len(pred) >= horizon else np.full(horizon, np.nan)
            else:
                last_val = last_train_vals.get(atm, 0)
                pred = np.full(horizon, last_val)
            
            mask = ~np.isnan(actual) & ~np.isnan(pred)
            
            if np.sum(mask) > 5:
                a_clean = actual[mask]
                p_clean = pred[mask]
                
                smape_list.append(smape_safe(a_clean, p_clean))
                mae_list.append(mean_absolute_error(a_clean, p_clean))
                rmse_list.append(np.sqrt(mean_squared_error(a_clean, p_clean)))
                bias_list.append(np.mean(p_clean - a_clean))
            else:
                smape_list.append(np.nan)
                mae_list.append(np.nan)
                rmse_list.append(np.nan)
                bias_list.append(np.nan)
        
        print(f"{name:<12} "
              f"{np.nanmean(smape_list):<8.2f} "
              f"{np.nanmean(mae_list):<8.2f} "
              f"{np.nanmean(rmse_list):<8.2f} "
              f"{np.nanmean(bias_list):<8.2f}")
    
    print("-" * 55)

## 7. Train ARIMA Baseline

In [19]:
def train_arima_baseline(train_df, test_df, stationarity_info):
    """
    Train ARIMA baseline model for all 111 ATMs.
    """
    predictions = {}
    smape_scores = []
    diff_applied = {}
    
    print("🚀 Training ARIMA Baseline...\n")
    
    train_wide = train_df.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
    test_wide = test_df.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
    
    failed_count = 0
    
    for atm in tqdm(train_wide.columns, desc="ARIMA training"):
        train_series = train_wide[atm].dropna()
        actual = test_wide[atm].values
        
        if len(train_series) < 10:
            failed_count += 1
            forecast = np.full(FORECAST_HORIZON, train_series.iloc[-1] if not train_series.empty else 0)
            predictions[atm] = forecast
            continue
            
        try:
            d = stationarity_info[atm]['d']
            diff_applied[atm] = d
            
            model = ARIMA(train_series, order=(5, d, 2),
                          enforce_stationarity=False, enforce_invertibility=False)
            fitted = model.fit()
            
            forecast = fitted.forecast(steps=FORECAST_HORIZON).values
            forecast = np.maximum(0, forecast)
            
            predictions[atm] = forecast
            
        except Exception as e:
            failed_count += 1
            last_val = train_series.iloc[-1]
            predictions[atm] = np.full(FORECAST_HORIZON, last_val)
            
    print(f"\n{'='*60}")
    print(f"🎯 ARIMA Baseline - Training Complete!")
    print(f"{'='*60}")
    print(f"   ✅ Success: {len(train_wide.columns) - failed_count}/{len(train_wide.columns)} ATMs")
    print(f"   ❌ Failed:  {failed_count} ATMs")
    print(f"{'='*60}\n")
    
    return predictions, diff_applied

# Train ARIMA
arima_predictions, arima_diff_applied = train_arima_baseline(train_ts, test_df, stationarity_info)

# Evaluate ARIMA
evaluate_model_comprehensive("ARIMA Baseline", train_ts, val_df, test_df, arima_predictions)

🚀 Training ARIMA Baseline...



ARIMA training: 100%|██████████| 111/111 [01:51<00:00,  1.01s/it]


🎯 ARIMA Baseline - Training Complete!
   ✅ Success: 111/111 ATMs
   ❌ Failed:  0 ATMs


=== ARIMA Baseline EVALUATION ===
Tập          SMAPE    MAE      RMSE     Bias    
-------------------------------------------------------
Val          39.63    7.13     9.10     -0.77   
Test         33.54    6.06     7.50     -0.33   
-------------------------------------------------------


## 8. Train SARIMAX with Exogenous Features

In [20]:
def train_sarimax_safe(train_df, test_df, stationarity_info):
    """
    Train SARIMAX with safety mechanisms (circuit breaker for explosive forecasts).
    """
    predictions = {}
    diff_applied = {}
    
    print("🚀 Training SARIMAX (Safe Mode)...\n")
    
    train_wide = train_df.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
    test_wide = test_df.pivot(index='Date', columns='ATM_ID', values='Withdrawal')
    
    # Create exogenous features
    def create_exog(index):
        exog = pd.DataFrame(index=index)
        exog['day_of_week'] = index.dayofweek
        exog['is_weekend'] = (exog['day_of_week'] >= 5).astype(int)
        exog['is_payday'] = index.day.isin([1, 15, 25, 28, 30, 31]).astype(int)
        return exog
    
    exog_train_full = create_exog(train_wide.index)
    exog_test = create_exog(test_wide.index)
    
    horizon = len(test_wide)
    failed_count = 0
    explosive_count = 0
    diff_applied_count = 0
    
    for atm in tqdm(train_wide.columns, desc="SARIMAX training"):
        train_series = train_wide[atm].dropna()
        
        if len(train_series) < 21:
            failed_count += 1
            last_val = train_series.iloc[-1] if not train_series.empty else 0
            predictions[atm] = np.full(horizon, last_val)
            diff_applied[atm] = 0
            continue
            
        try:
            exog_train = exog_train_full.loc[train_series.index]
            d = stationarity_info[atm]['d']
            
            if d == 1:
                diff_applied_count += 1
            
            model = SARIMAX(
                train_series,
                exog=exog_train,
                order=(3, d, 2),
                seasonal_order=(1, 1, 1, 7)
            )
            fitted = model.fit(disp=False)
            
            forecast = fitted.forecast(steps=horizon, exog=exog_test).values
            
            # Circuit breaker: Check for explosive forecasts
            historical_max = train_series.max()
            if np.max(forecast) > historical_max * 3:
                explosive_count += 1
                raise ValueError("Explosive forecast detected!")
            
            # Clip to safe bounds
            safe_upper_bound = historical_max * 1.5
            forecast = np.clip(forecast, 0, safe_upper_bound)
            
            predictions[atm] = forecast
            diff_applied[atm] = d
            
        except Exception as e:
            failed_count += 1
            last_val = train_series.iloc[-1]
            predictions[atm] = np.full(horizon, last_val)
            diff_applied[atm] = stationarity_info[atm]['d']
    
    print(f"\n{'='*60}")
    print(f"🎯 SARIMAX (Safe Mode) - Training Complete!")
    print(f"{'='*60}")
    print(f"   ✅ Success (Stable)  : {len(train_wide.columns) - failed_count}/{len(train_wide.columns)} ATMs")
    print(f"   ❌ Failed (Fallback) : {failed_count} ATMs")
    print(f"   🧨 Explosive halted  : {explosive_count} ATMs")
    print(f"   🔄 Differencing (d=1): {diff_applied_count} ATMs")
    print(f"{'='*60}\n")
    
    return predictions, diff_applied

# Train SARIMAX
sarimax_predictions, sarimax_diff_applied = train_sarimax_safe(train_ts, test_df, stationarity_info)

# Evaluate SARIMAX
evaluate_model_comprehensive("SARIMAX (Safe Mode)", train_ts, val_df, test_df, sarimax_predictions)

🚀 Training SARIMAX (Safe Mode)...



SARIMAX training: 100%|██████████| 111/111 [04:07<00:00,  2.23s/it]


🎯 SARIMAX (Safe Mode) - Training Complete!
   ✅ Success (Stable)  : 111/111 ATMs
   ❌ Failed (Fallback) : 0 ATMs
   🧨 Explosive halted  : 0 ATMs
   🔄 Differencing (d=1): 10 ATMs


=== SARIMAX (Safe Mode) EVALUATION ===
Tập          SMAPE    MAE      RMSE     Bias    
-------------------------------------------------------
Val          32.70    5.81     8.36     -0.16   
Test         19.72    3.29     4.62     0.29    
-------------------------------------------------------


## 9. Train Global LightGBM Model

In [21]:
def train_lightgbm_global(train_feat, val_feat, test_feat):
    """
    Train 1 global LightGBM model for all 111 ATMs using ATM_ID as categorical feature.
    """
    print(f"🚀 Training Global LightGBM (Optimized)...\n")
    
    # Prepare data
    df_train = train_feat.copy()
    df_val = val_feat.copy()
    df_test = test_feat.copy()
    
    for df in [df_train, df_val, df_test]:
        df['ATM_ID'] = df['ATM_ID'].astype('category')
    
    # Define features
    feature_cols = [col for col in df_train.columns if col not in ['Date', 'Withdrawal']]
    
    X_train = df_train[feature_cols]
    y_train = df_train['Withdrawal']
    X_val = df_val[feature_cols]
    y_val = df_val['Withdrawal']
    X_test = df_test[feature_cols]
    
    # Create LightGBM datasets
    train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=['ATM_ID'], free_raw_data=False)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set, categorical_feature=['ATM_ID'], free_raw_data=False)
    
    # Hyperparameters
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'learning_rate': 0.03,
        'max_depth': 7,
        'num_leaves': 63,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_data_in_leaf': 20,
        'verbose': -1,
        'seed': 42,
        'n_jobs': -1
    }
    
    # Train model
    model = lgb.train(
        params,
        train_set,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100)
        ],
        num_boost_round=1500
    )
    
    # Make predictions
    raw_predictions = model.predict(X_test)
    df_test['Pred'] = raw_predictions
    
    # Package results
    predictions = {}
    for atm in df_test['ATM_ID'].unique():
        atm_data = df_test[df_test['ATM_ID'] == atm]
        pred_values = np.maximum(0, atm_data['Pred'].values)
        predictions[atm] = pred_values
    
    print(f"\n{'='*55}")
    print(f"🎯 Global LightGBM - Training Complete!")
    print(f"{'='*55}")
    print(f"   ✅ Trained: 1 Global Model for {len(predictions)} ATMs")
    print(f"{'='*55}\n")
    
    return predictions, model

# Train LightGBM
lgb_predictions, lgb_model = train_lightgbm_global(train_ml, val_feat, test_feat)

# Evaluate LightGBM
evaluate_model_comprehensive("Global LightGBM (Optimized)", train_ml, val_df, test_df, lgb_predictions)

🚀 Training Global LightGBM (Optimized)...



[100]	valid_0's l1: 4.52307
[200]	valid_0's l1: 3.96596
[300]	valid_0's l1: 3.90943
[400]	valid_0's l1: 3.87638
[500]	valid_0's l1: 3.85253

🎯 Global LightGBM - Training Complete!
   ✅ Trained: 1 Global Model for 111 ATMs


=== Global LightGBM (Optimized) EVALUATION ===
Tập          SMAPE    MAE      RMSE     Bias    
-------------------------------------------------------
Val          33.44    5.92     8.55     -0.74   
Test         17.31    2.83     4.10     -0.28   
-------------------------------------------------------


## 10. Feature Importance (SHAP Analysis)

In [22]:
def analyze_shap_importance(model, test_feat, output_dir='../report/figures'):
    """
    Analyze LightGBM feature importance using SHAP.
    """
    print("🔍 Computing SHAP values (may take 1-2 minutes)...\n")
    
    # Prepare data
    feature_cols = [col for col in test_feat.columns if col not in ['Date', 'Withdrawal']]
    X_test = test_feat[feature_cols].copy()
    X_test['ATM_ID'] = X_test['ATM_ID'].astype('category')
    
    # Sample for faster computation
    X_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)
    
    # Initialize TreeExplainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # Plot 1: Summary Plot
    print("📊 1. SHAP Summary Plot")
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample, max_display=15, show=False)
    plt.title("Feature Impact on Withdrawal Demand", fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/07_shap_summary.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   ✅ Saved: 07_shap_summary.png")
    
    # Plot 2: Bar Plot
    print("\n📊 2. SHAP Bar Plot (Feature Importance)")
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=15, show=False)
    plt.title("Top 15 Most Important Features", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/08_shap_importance.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   ✅ Saved: 08_shap_importance.png")
    
    # Plot 3: Dependence Plot (lag_7)
    if 'lag_7' in X_sample.columns:
        print("\n📊 3. SHAP Dependence Plot (lag_7)")
        plt.figure(figsize=(8, 6))
        shap.dependence_plot("lag_7", shap_values, X_sample, show=False)
        plt.title("Non-linear Dependency of lag_7", fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/09_shap_dependence_lag7.png', dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"   ✅ Saved: 09_shap_dependence_lag7.png")
    
    print(f"\n✅ SHAP analysis complete")

# Run SHAP analysis
analyze_shap_importance(lgb_model, test_feat)

🔍 Computing SHAP values (may take 1-2 minutes)...

📊 1. SHAP Summary Plot
   ✅ Saved: 07_shap_summary.png

📊 2. SHAP Bar Plot (Feature Importance)
   ✅ Saved: 08_shap_importance.png

📊 3. SHAP Dependence Plot (lag_7)
   ✅ Saved: 09_shap_dependence_lag7.png

✅ SHAP analysis complete


<Figure size 800x600 with 0 Axes>

## 11. Visualization: Actual vs Predicted

In [27]:
def plot_actual_vs_predicted(test_df, predictions_dict, model_name, output_dir='../report/figures', atm_id=None):
    """
    Plot actual vs predicted values for a sample ATM.
    """
    if atm_id is None:
        atm_id = list(predictions_dict.keys())[0]
    
    atm_data = test_df[test_df['ATM_ID'] == atm_id].copy().sort_values('Date')
    pred_values = predictions_dict[atm_id]
    
    fig, ax = plt.subplots(figsize=(14, 5.5))
    
    # Highlight weekends
    weekends = atm_data[atm_data['Date'].dt.dayofweek >= 5]['Date']
    for we in weekends:
        ax.axvspan(we - pd.Timedelta(hours=12), we + pd.Timedelta(hours=12),
                   color=BG_HIGHLIGHT, alpha=0.6, lw=0, zorder=0)
    
    # Plot actual (background)
    ax.plot(atm_data['Date'], atm_data['Withdrawal'], color='#a6a6a6',
            marker='o', markersize=4, linestyle='-', linewidth=1.5, zorder=2)
    
    # Plot prediction (highlight)
    ax.plot(atm_data['Date'], pred_values, color=MAIN_BLUE,
            marker='None', linestyle='-', linewidth=2.5, zorder=3)
    
    # Direct labels
    last_date = atm_data['Date'].iloc[-1]
    last_actual = atm_data['Withdrawal'].iloc[-1]
    last_pred = pred_values[-1]
    offset = 1.5 if last_pred >= last_actual else -1.5
    
    ax.text(last_date + pd.Timedelta(days=1), last_actual - offset, 'Actual',
            color='#a6a6a6', fontsize=12, fontweight='bold', va='center')
    ax.text(last_date + pd.Timedelta(days=1), last_pred + offset, f'{model_name}\nForecast',
            color=MAIN_BLUE, fontsize=12, fontweight='bold', va='center')
    
    # Formatting
    ax.set_title(f"{model_name} - 56-day Forecast (ATM: {atm_id})",
                 fontsize=14, fontweight='bold', color=MAIN_BLUE, loc='left', pad=20)
    ax.set_xlabel('')
    ax.set_ylabel('Withdrawal Volume', fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
    
    sns.despine(ax=ax)
    ax.yaxis.grid(True, linestyle='-', color='#eeeeee', zorder=1)
    ax.set_xlim(atm_data['Date'].iloc[0] - pd.Timedelta(days=1),
                last_date + pd.Timedelta(days=6))
    
    plt.tight_layout()
    filename = f"{output_dir}/10_{model_name.lower().replace(' ', '_')}_actual_vs_pred.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✅ Saved: {filename.split('/')[-1]}")

# Plot results for all models
print("📊 Generating Actual vs Predicted plots...\n")

sample_atm = test_feat['ATM_ID'].unique()[0]

plot_actual_vs_predicted(test_feat, arima_predictions, "ARIMA", atm_id=sample_atm)
plot_actual_vs_predicted(test_feat, sarimax_predictions, "SARIMAX", atm_id=sample_atm)
plot_actual_vs_predicted(test_feat, lgb_predictions, "LightGBM", atm_id=sample_atm)

print(f"\n✅ All visualizations saved to report/figures/")

📊 Generating Actual vs Predicted plots...

✅ Saved: 10_arima_actual_vs_pred.png
✅ Saved: 10_sarimax_actual_vs_pred.png
✅ Saved: 10_lightgbm_actual_vs_pred.png

✅ All visualizations saved to report/figures/


## 12. Save Model Predictions

In [ ]:
print("💾 Saving model predictions...\n")

# Save predictions as dictionaries
import pickle

with open('../outputs/arima_predictions.pkl', 'wb') as f:
    pickle.dump(arima_predictions, f)

with open('../outputs/sarimax_predictions.pkl', 'wb') as f:
    pickle.dump(sarimax_predictions, f)

with open('../outputs/lgb_predictions.pkl', 'wb') as f:
    pickle.dump(lgb_predictions, f)

print("✅ Successfully saved predictions:")
print("   - data/processed/arima_predictions.pkl")
print("   - data/processed/sarimax_predictions.pkl")
print("   - data/processed/lgb_predictions.pkl")

💾 Saving model predictions...

✅ Successfully saved predictions:
   - data/processed/arima_predictions.pkl
   - data/processed/sarimax_predictions.pkl
   - data/processed/lgb_predictions.pkl
